In [ ]:
import sys
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset, Subset
from torchvision.utils import make_grid
import torch.nn.functional as F


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random


import models as models
import train_helper as train_helper
import utils as utils
import data_helper as data_helper

vae_path = "/home/benjiy/repo/Verified-Synthetic-Data/MNIST"
sys.path.append(vae_path)

In [ ]:
# Set up device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_seed = 0
torch.manual_seed(base_seed)
torch.cuda.manual_seed_all(base_seed)
np.random.seed(base_seed)
random.seed(base_seed)

model_saved_path = os.path.join(os.getcwd(),"model_saved")
data_saved_path = os.path.join(os.getcwd(),"data_saved")
results_saved_path = os.path.join(os.getcwd(),"results_saved")




## Prepare data ##


In [ ]:


full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())

test_dataset = datasets.MNIST(root="./data", train=False, download=True,transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
full_digit_indices = utils.create_balanced_subset_indices(full_dataset,seed=base_seed)

#train_dataset_5000 = Subset(full_dataset, range(5000))


## Load/train small model

In [ ]:
init_size = 5000

init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
init_dataset = Subset(full_dataset, init_subset)
train_loader_5000 = DataLoader(init_dataset, batch_size=128, shuffle=True)
model_5000 = models.CVAE(input_dim=784, label_dim=10,latent_dim=20,name="cvae_conv_5000",arch="conv").to(device)
#train_logs = training.train_model(model, train_loader_5000, device, epochs=200, lr=1e-3, patience=5)
train_helper.train_model_with_validation(model=model_5000, train_loader=train_loader_5000, val_loader=test_loader, device=device, epochs=200, lr=1e-3, patience=5)
# utils.save_model(model_5000, "cvae_real_5000_new", model_saved_path)

# model_5000 = utils.load_model("cvae_real_5000_new", model_saved_path , device, (784, 10, 20))


In [ ]:
#utils.plot_samples_per_digit(8, model_5000)

### Load/Train Full model

In [ ]:

 
train_loader_60000 = DataLoader(full_dataset, batch_size=128, shuffle=True)
model_60000 = models.CVAE(input_dim=784, label_dim=10,latent_dim=20).to(device)
train_logs_60000 = train_helper.train_model_with_validation(model_60000, train_loader_60000, test_loader, device, epochs=200, lr=1e-3, patience=5)
utils.save_model(model_60000, "cvae_real_60000_new", model_saved_path)

#model_60000 = utils.load_model("cvae_real_60000_new", model_saved_path , device, (784, 10, 20))


## Train Discriminator


In [ ]:
# Train Discriminator
discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, model_5000, device)
disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
discmodel_5000 = models.SyntheticDiscriminator(input_dim=784)
disc_logs_5000 = train_helper.train_model(model=discmodel_5000, train_loader=disc_loader, device=device, epochs=50, lr=1e-3, patience=5)
utils.save_model(discmodel_5000, "disc_5000_sample60000_new", model_saved_path)

# discmodel_5000 = utils.load_model("disc_5000_sample60000_new", model_saved_path , device, (784,))


## Retrain with synthetic data

### generate synthetic data

In [ ]:
#data_helper.generate_images_with_filtering(model_5000, data_saved_path, "cvae_real_5000_new", total_samples=240000, batch_size=60000, discriminator=discmodel_5000, selection_percentile=1, per_digit_filtering=True)

data_helper.generate_balanced_images_with_filtering(model=model_5000, save_directory=data_saved_path, model_name="cvae_real_5000_new",
 total_samples=300000, batch_size=60000, discriminator=discmodel_5000, selection_threshold=0.5, use_quantile_filtering=False)

### check synthetic data

In [ ]:
utils.display_samples_from_pt_file(5,os.path.join(data_saved_path,"cvae_real_5000_new","cvae_real_5000_new_60000_t0.5_b0.pt"))

### retrain with synthetic data

In [ ]:
synthetic_data_load_path = os.path.join(data_saved_path,"cvae_real_5000_new")
synthetic_train_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20).to(device)
utils.verify_balance(synthetic_train_loader.dataset)

In [ ]:

synthetic_train_logs = train_helper.train_model_with_validation(synthetic_model, synthetic_train_loader, test_loader, device, epochs=200, lr=1e-3, patience=5,verbose=True)
#synthetic_train_logs = train_helper.train_model(synthetic_model, synthetic_train_loader, device, epochs=200, lr=1e-3, patience=5,verbose=True)

In [ ]:
print(train_helper.calculate_validation_loss(model_5000, test_loader, device))
# print(train_helper.calculate_validation_loss(model_60000, test_loader, device))
print(train_helper.calculate_validation_loss(synthetic_model, test_loader, device))


## Train model with different sizes

In [ ]:
model_sizes = [1000, 2000, 5000, 10000, 20000, 30000, 40000, 50000, 60000]
all_logs = []
all_models = []
test_error = {"val_loss":[], "kl":[], "recon":[]}
for model_size in model_sizes:
    balanced_subset = utils.get_balanced_subset(full_digit_indices, model_size)
    train_dataset_balanced = Subset(full_dataset, balanced_subset)
    train_loader_balanced = DataLoader(train_dataset_balanced, batch_size=128, shuffle=True)
    model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20, name=f"cvae_{model_size}_balanced").to(device)
    train_logs = train_helper.train_model_with_validation(model, train_loader_balanced, test_loader, device, epochs=200, lr=1e-3, patience=5,verbose=False)
    utils.save_model(model=model, model_name=model.name, path=model_saved_path)
    val_loss, val_kl, val_recon = train_helper.calculate_validation_loss(model, test_loader, device)
    test_error["val_loss"].append(val_loss)
    test_error["kl"].append(val_kl)
    test_error["recon"].append(val_recon)
    all_logs.append(train_logs)
    all_models.append(model)
    print(f"Model size: {model_size}, Val Loss: {val_loss}, Val KL: {val_kl}, Val Recon: {val_recon}")



In [ ]:
len(all_models)
test_error = {"val_loss":[], "kl":[], "recon":[]}
for model in all_models:
    val_loss, val_kl, val_recon = train_helper.calculate_validation_loss(model, test_loader, device)
    test_error["val_loss"].append(val_loss)
    test_error["kl"].append(val_kl)
    test_error["recon"].append(val_recon)    

test_error['model_size'] = model_sizes
res_table = pd.DataFrame.from_dict(test_error, orient='columns')
print(res_table)
res_table.to_csv(os.path.join(results_saved_path, 'diff_modelsize_results.csv'), index=False)


## Repeat single step retraining process with different seeds

In [ ]:
model_size = 5000
synthetic_size = 60000
test_results = {"original_loss":[], "original_kl":[], "original_recon":[], "synthetic_loss":[], "synthetic_kl":[], "synthetic_recon":[]}
orig_subset = utils.get_balanced_subset(full_digit_indices, model_size)
orig_dataset = Subset(full_dataset, orig_subset)
orig_loader = DataLoader(orig_dataset, batch_size=128, shuffle=True)
num_experiments = 10
for exp_num in range(num_experiments):
    print(f"\n=== Experiment {exp_num + 1}/{num_experiments} ===")

    exp_seed = base_seed + exp_num
    torch.manual_seed(exp_seed)
    torch.cuda.manual_seed_all(exp_seed)
    np.random.seed(exp_seed)
    random.seed(exp_seed)

    # Train Original Model
    orig_model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20, name=f"cvae_{model_size}_repeat_{exp_seed}").to(device)
    train_helper.train_model(orig_model, orig_loader, device, epochs=200, lr=1e-3, patience=5)    
    val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(orig_model, test_loader, device)
    test_results["original_loss"].append(val_loss)
    test_results["original_kl"].append(val_kl)
    test_results["original_recon"].append(val_recon)

    # Train Discriminator
    discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, orig_model, device)
    disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
    disc_model = models.SyntheticDiscriminator(input_dim=784)    
    train_helper.train_model(disc_model, disc_loader, device, epochs=80, lr=1e-3, patience=5)

    # Generate Synthetic Data
    data_helper.generate_images_with_filtering(orig_model, data_saved_path, orig_model.name, 
    total_samples=synthetic_size, batch_size=60000, discriminator=disc_model, selection_percentile=1, per_digit_filtering=True, verbose=False)

    # Train Synthetic Model
    synthetic_data_load_path = os.path.join(data_saved_path,orig_model.name)
    synthetic_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
    synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20).to(device)
    train_helper.train_model(synthetic_model, synthetic_loader, device, epochs=200, lr=1e-3, patience=5)
    val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(synthetic_model, test_loader, device)
    test_results["synthetic_loss"].append(val_loss)
    test_results["synthetic_kl"].append(val_kl)
    test_results["synthetic_recon"].append(val_recon)

res_table = pd.DataFrame.from_dict(test_results, orient='columns')


In [ ]:
summary_stats = {
    'Metric': ['Loss', 'KL Divergence', 'Reconstruction'],
    'Original_Mean': [
        np.mean(test_results["original_loss"]),
        np.mean(test_results["original_kl"]),
        np.mean(test_results["original_recon"])
    ],
    'Original_Std': [
        np.std(test_results["original_loss"]),
        np.std(test_results["original_kl"]),
        np.std(test_results["original_recon"])
    ],
    'Synthetic_Mean': [
        np.mean(test_results["synthetic_loss"]),
        np.mean(test_results["synthetic_kl"]),
        np.mean(test_results["synthetic_recon"])
    ],
    'Synthetic_Std': [
        np.std(test_results["synthetic_loss"]),
        np.std(test_results["synthetic_kl"]),
        np.std(test_results["synthetic_recon"])
    ],
    'Improvement_Mean': [
        np.mean([o - s for o, s in zip(test_results["original_loss"], test_results["synthetic_loss"])]),
        np.mean([o - s for o, s in zip(test_results["original_kl"], test_results["synthetic_kl"])]),
        np.mean([o - s for o, s in zip(test_results["original_recon"], test_results["synthetic_recon"])])
    ],
    'Improvement_Std': [
        np.std([o - s for o, s in zip(test_results["original_loss"], test_results["synthetic_loss"])]),
        np.std([o - s for o, s in zip(test_results["original_kl"], test_results["synthetic_kl"])]),
        np.std([o - s for o, s in zip(test_results["original_recon"], test_results["synthetic_recon"])])
    ]
}

summary_df = pd.DataFrame(summary_stats)
print(summary_df)

## Iterative Re-Training

In [ ]:
init_size = 30000

test_results = {"val_loss":[], "val_recon":[], "val_kl":[]}
init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
init_dataset = Subset(full_dataset, init_subset)
init_loader = DataLoader(init_dataset, batch_size=128, shuffle=True)


this_model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20, name=f"cvae_iter_{init_size}").to(device)
train_helper.train_model(this_model, init_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)
val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(this_model, test_loader, device)
test_results["val_loss"].append(val_loss)
test_results["val_recon"].append(val_recon)
test_results["val_kl"].append(val_kl)
utils.save_model(this_model, this_model.name, model_saved_path)

synthetic_size_schedule = [60000, 100000, 150000, 200000, 300000]
for synthetic_size in synthetic_size_schedule:
    print(f"\n=== Synthetic Size: {synthetic_size} ===")
    # Train Discriminator
    discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, this_model, device)
    disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)

    discriminator_test_dataset = data_helper.prepare_discriminator_dataset(test_dataset, this_model, device)
    disc_test_loader = DataLoader(discriminator_test_dataset, batch_size=128, shuffle=True)

    disc_model = models.SyntheticDiscriminator(input_dim=784)    
    train_helper.train_model_with_validation(disc_model, disc_loader, disc_test_loader, device, epochs=80, lr=1e-3, patience=5, verbose=False)

    # Generate Synthetic Data
    data_helper.generate_balanced_images_with_filtering(model=this_model, save_directory=data_saved_path, model_name=this_model.get_name(), 
    total_samples=synthetic_size, discriminator=disc_model, selection_threshold=0.5, verbose=False)

    # Train Synthetic Model
    synthetic_data_load_path = os.path.join(data_saved_path,this_model.name)
    synthetic_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
    synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20, name=f"cvae_iter_{synthetic_size}").to(device)
    train_helper.train_model(synthetic_model, synthetic_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)
    val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(synthetic_model, test_loader, device)
    test_results["val_loss"].append(val_loss)
    test_results["val_recon"].append(val_recon)
    test_results["val_kl"].append(val_kl)

    print(f"Synthetic Size: {synthetic_size} - Val Loss: {val_loss:.4f} - Val KL: {val_kl:.4f} - Val Recon: {val_recon:.4f}")

    utils.save_model(synthetic_model, synthetic_model.get_name(), model_saved_path)
    this_model = synthetic_model

    del synthetic_loader
    del disc_model
    del discriminator_dataset
    del disc_loader
    

res_table = pd.DataFrame.from_dict(test_results, orient='columns')


In [ ]:
res_table

## Training results check

In [ ]:
utils.plot_samples_per_digit(5,synthetic_model)

In [ ]:
utils.plot_samples_per_digit(5,model_5000)


# check discriminator improvement

In [ ]:
disc_dataset_base = data_helper.prepare_discriminator_dataset(full_dataset, model_5000, device)
disc_loader_base = DataLoader(disc_dataset_base, batch_size=128, shuffle=True)
disc_test_dataset_base = data_helper.prepare_discriminator_dataset(test_dataset, model_5000, device)
disc_test_loader_base = DataLoader(disc_test_dataset_base, batch_size=128, shuffle=True)

discmodel_5000_base = models.SyntheticDiscriminator(input_dim=784)
train_helper.train_model_with_validation(model=discmodel_5000_base, train_loader=disc_loader_base, val_loader=disc_test_loader_base, device=device, epochs=100, lr=1e-3, patience=5)


In [ ]:
disc_dataset_label = data_helper.prepare_discriminator_dataset_with_labels(full_dataset, model_5000, device)
disc_loader_label = DataLoader(disc_dataset_label, batch_size=128, shuffle=True)
disc_test_dataset_label = data_helper.prepare_discriminator_dataset_with_labels(test_dataset, model_5000, device)
disc_test_loader_label = DataLoader(disc_test_dataset_label, batch_size=128, shuffle=True)




In [ ]:
discmodel_5000_mlp = models.ConditionalDiscriminator(input_dim=784, name="disc_5000_mlp", arch="mlp")
train_helper.train_model_with_validation(model=discmodel_5000_mlp, train_loader=disc_loader_label, val_loader=disc_test_loader_label, device=device, epochs=200, lr=1e-3, patience=5)

In [ ]:
discmodel_5000_conv = models.ConditionalDiscriminator(input_dim=784, name="disc_5000_conv", arch="conv")
train_helper.train_model_with_validation(model=discmodel_5000_conv, train_loader=disc_loader_label, val_loader=disc_test_loader_label, device=device, epochs=200, lr=1e-3, patience=5)

## Try retraining with better discriminator

In [ ]:
all_models = []

In [ ]:
all_models

In [ ]:
import shutil

test_results = {"val_loss":[], "val_recon":[], "val_kl":[], "model_name":[]}

# init_size = 5000
# init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
# init_dataset = Subset(full_dataset, init_subset)
# init_loader = DataLoader(full_dataset, batch_size=128, shuffle=True)


# this_model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20, name=f"cvae_conv_real_full",arch="conv").to(device)
# train_helper.train_model(this_model, init_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)

# this_model = utils.load_model(model_name="cvae_conv_real_5000", path=model_saved_path, input_device=device, model_args=(784, 10, 20, "conv_conv_real_5000", "conv"))
this_model = all_models[-1]
val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(this_model, test_loader, device)
test_results["val_loss"].append(val_loss)
test_results["val_recon"].append(val_recon)
test_results["val_kl"].append(val_kl)
test_results["model_name"].append(this_model.name)


print(f"Model: {this_model.get_name()} - Val Loss: {val_loss:.4f} - Val KL: {val_kl:.4f} - Val Recon: {val_recon:.4f}")
# all_models.append(this_model)

discriminator_dataset = data_helper.prepare_discriminator_dataset_with_labels(full_dataset, this_model, device)
disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
discriminator_test_dataset = data_helper.prepare_discriminator_dataset_with_labels(test_dataset, this_model, device)
disc_test_loader = DataLoader(discriminator_test_dataset, batch_size=128, shuffle=True)
disc_model = models.ConditionalDiscriminator(input_dim=784, name="disc_5000_conv", arch="conv")
#train_helper.train_model(model=disc_model, train_loader=disc_loader, device=device, epochs=200, lr=1e-3, patience=5, verbose=True)
train_logs = train_helper.train_model_with_validation(model=disc_model, train_loader=disc_loader, val_loader=disc_test_loader, device=device, epochs=200, lr=1e-3, patience=5, verbose=True)




In [ ]:

# utils.save_model(this_model, this_model.get_name(), model_saved_path)

for threshold in [0.2,0.8,0.1,0.5]:
    for synthetic_size in [200_000,500_000,1_000_000]:
        # Generate Synthetic Data
        model_name = f'cvae_conv_disc_conv_q{threshold}_{synthetic_size}_on_conv_q0.2_200000'
        synthetic_data_load_path = os.path.join(data_saved_path,model_name)
        data_helper.generate_balanced_images_with_filtering(model=this_model, save_directory=synthetic_data_load_path, 
        total_samples=synthetic_size, discriminator=disc_model, selection_threshold=threshold, verbose=False, use_quantile_filtering=True)

        # Train Synthetic Model

        synthetic_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
        synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20, name=model_name,arch="conv").to(device)
        train_helper.train_model(synthetic_model, synthetic_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)
        val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(synthetic_model, test_loader, device)
        test_results["val_loss"].append(val_loss)
        test_results["val_recon"].append(val_recon)
        test_results["val_kl"].append(val_kl)
        test_results["model_name"].append(synthetic_model.get_name())

        print(f"Model Name: {model_name} - Val Loss: {val_loss:.4f} - Val KL: {val_kl:.4f} - Val Recon: {val_recon:.4f}")

        utils.save_model(synthetic_model, synthetic_model.get_name(), model_saved_path)
        all_models.append(synthetic_model)

        del synthetic_loader

res_table = pd.DataFrame.from_dict(test_results, orient='columns')

In [ ]:
res_table.to_csv(os.path.join(results_saved_path, 'disc_conv_onestep_q_combine_size_results.csv'),index=False,header=False,mode='a')
print(res_table)